https://github.com/thor102/deliverable_3

Har brukt generativ AI til finpusse kode ut ifra utkast som vi har gitt til den. (Se sammendrag på canvas)

reference: <br>
orkslayergamedev, Python Classes Text Battle, GitHub, c. 2023, https://github.com/orkslayergamedev/python-classes-text-battle (accessed 03.10.2026).

Task 1: Create a Character class that allows characters to take damage, heal themselves, and check whether they are still alive.

In [ ]:
class Character:
    """Represents a character in the RPG"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon = None) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100).
            damage (int): 	Damage dealt when the character attacks (default = 10).
            weapon (Weapon): Optional weapon equipped by the character (default = None).
        """
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.weapon = weapon
        
    def attack(self, target) -> None:
        """Attack and give damage value to target character."""
        if self.weapon is not None:
            damage_done = self.weapon.roll_damage()
        else:
            damage_done = self.damage
        target.take_damage(damage_done)


    def take_damage(self, damage_done: int) -> None:
        """Takes damage from attack and takes it from hp amount."""
        self._hp -= abs(damage_done)
        self._hp = max(self._hp, 0)  # Target hp cannot end up lower than 0


    def heal(self, healing: int = 20) -> None:
        """Restore HP without exceeding the maximum HP."""
        self._hp += abs(healing) # abs makes sure that user can't use negtaive values.
        self._hp = min(self.hp_max, self._hp)

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp
    
    @property 
    def is_alive(self) -> bool:
        """Check whether the character is still alive."""
        if self.hp > 0:
            return True 
        else:
            return False 


In [ ]:
# Test for task 1

hero = Character("Hero")
hero.take_damage(30)
hero.heal() # heals 20 hp
print("hero hp after damage and healing:", hero.hp)
print("is hero still alive?", hero.is_alive)
hero.take_damage(100) # Shows that .is_alive works
print("is hero alive?", hero.is_alive)


hero hp after damagae and healing: 90
is hero still alive? True
is hero alive? False


**Task 1 Discussion** <br>
The Character class allows characters to receive damage and heal themselves. The HP value is kept between 0 and the maximum HP, and the is_alive property checks whether the character is still alive.

Task 2: The Dice class creates dice with a specified number of sides. If no number is provided, the default is 6 sides. The class can roll the dice and return a random number between 1 and the number of sides.

In [ ]:
import random 


class Dice:
    """Represents a dice to make it more random what happens."""
    def __init__(self, sides: int = 6) -> None:
        """       
        Initialise the Dice
        Args:
            sides (int): The amount of sides on the dice (default 6)."""
        self.sides= max(sides, 2)


    def roll(self) -> int:
        """returns random value from 1 to 6 or the value the user sets."""
        return random.randint(1, self.sides)


    def __repr__(self) -> str:
        """Return a string representation of the Dice object."""
        return f"Dice(sides={self.sides})"     


    def __call__(self) -> int:
        """Allow the Dice object to be called directly to roll the dice."""
        return self.roll()        


In [ ]:
# Test for task 2 
 
d20 = Dice(sides=20)
print(d20)

# Shows that both versions work 
print(d20.roll())
print(d20())


Dice(sides=20)
10
6


**Task 2 Discussion** <br>
The Dice class creates a dice with 6 sides by default, but the user can choose a different number of sides. The class also includes a __call__() method, which makes it easier to roll the dice by calling the object directly. 

Task 3: The Weapon class creates weapons that characters can use in battle. When a character has a weapon equipped, it automatically uses the weapon when attacking. The damage is determined by a random dice roll from the Dice class, instead of the character's default damage.

In [ ]:
from dice import Dice


class Weapon:
    """Represents a weapon using dice for damage."""
    def __init__(self, name: str, damage_dice: Dice) -> None:  
        """              
        Initialise the Weapon
        Args:
            name (str): The name of the weapon. 
            damage_dice (Dice) The Dice object used to determine weapon damage."""
        self.name = name
        self.damage_dice = damage_dice 


    def roll_damage(self) -> int:
        """Return weapon damage based on a dice roll."""
        return self.damage_dice.roll() 


In [243]:
# Test for task 3

sword = Weapon("sword of the morning", Dice(20))
print(sword.roll_damage())


20


**Task 3 Discussion** <br>
One limitation of the Weapon class is that a weapon can deal less damage than an attack without a weapon. This could frustrate players and make weapons less useful. A possible improvement would be to add the weapon damage to the character's base damage instead of replacing it.

Task 4: The Character class is extended with two specialised subclasses, Warrior and Mage. Each subclass has its own abilities that affect how the character performs in battle.

In [ ]:
class Warrior(Character):
    """A heavily armored character."""

    def __init__(self, name: str, armor: int = 5, **kwargs): 
        super().__init__(name=name, **kwargs)
        self.armor = armor

    def take_damage(self, damage_done: int) -> None:
        """Reduce incoming damage by armor."""
        reduced_damage = max(damage_done - self.armor, 0)
        super().take_damage(reduced_damage)

    def __str__(self) -> str:
        return (
            f"Warrior: {self.name}, "
            f"HP={self.hp}, "
            f"Damage={self.damage}, "
            f"Armor={self.armor}"
        )

class Mage(Character):
    """A spellcaster that uses mana."""

    def __init__(
        self,
        name: str,
        mana: int = 30,
        mana_max: int = 30,
        **kwargs
    ):
        super().__init__(name=name, **kwargs)

        self.mana_max = mana_max
        self.mana = min(mana, mana_max)

    def attack(self, target) -> None:
        """Cast a spell if enough mana, otherwise use a normal attack."""

        if self.mana >= 10:
            self.mana -= 10
            target.take_damage(2 * self.damage)

        else:
            super().attack(target)
            self.mana = min(self.mana + 5, self.mana_max)

    def __str__(self) -> str:
        return (
            f"Mage: {self.name}, "
            f"HP={self.hp}, "
            f"Damage={self.damage}, "
            f"Mana={self.mana}/{self.mana_max}"
        )

In [ ]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon

# --- Warrior ---
w = Warrior("Bjorn")
assert (w.armor, w.hp, w.hp_max) == (5, 100, 100), "Warrior defaults"
w.take_damage(12)
assert w.hp == 93, "Armor should reduce 12 -> 7"
w.take_damage(2)
assert w.hp == 93, "Damage below armor should do 0 (not heal)"
w.take_damage(1000)
assert w.hp == 0 and not w.is_alive, "HP cannot go below 0"

w2 = Warrior("Tank", armor=3, hp_max=120, damage=15)
assert (w2.armor, w2.hp, w2.damage) == (3, 120, 15), "kwargs passed to Character"

# --- Mage: spell / fallback cycle ---
m = Mage("Merlin")
assert (m.mana, m.mana_max) == (30, 30), "Mage defaults"
dummy = Character("Dummy", hp_max=1000)
for _ in range(3):
    m.attack(dummy)                      # 3 spells: 3 * 2 * 10 = 60
assert dummy.hp == 940 and m.mana == 0, "Spells cost 10 mana, double damage"
m.attack(dummy)                          # fallback: 10 dmg, +5 mana
assert dummy.hp == 930 and m.mana == 5, "Fallback restores 5 mana"
m.attack(dummy); m.attack(dummy)         # fallback (mana 10), then spell
assert dummy.hp == 900 and m.mana == 0

# --- Mage + weapon ---
staff = Weapon("Staff", Dice(sides=2))
t = Character("Target")
Mage("A", weapon=staff).attack(t)
assert t.hp == 80, "Spell ignores weapon"
t = Character("Target")
m_low = Mage("B", mana=0, weapon=staff)
m_low.attack(t)
assert 98 <= t.hp <= 99 and m_low.mana == 5, "Fallback uses weapon"

assert Mage("C", mana=99).mana == 30, "Mana capped at mana_max"

# --- Polymorphism: spell goes through Warrior armor ---
w3 = Warrior("Shield")
Mage("D").attack(w3)
assert w3.hp == 85, "20 spell damage - 5 armor = 15"

assert isinstance(w3, Character) and isinstance(m, Character)
print(w3); print(m)
print("All Task 4 tests passed")

Warrior: Conan, HP=120, Damage=15, Armor=4
Mage: Merlin, HP=80, Damage=10, Mana=Merlin
Conan HP after attack: 99
Merlin mana left: 20


Task 5: Create a Battle class that is running battles to one of the character have died.

In [ ]:
class Battle: 
    """Simulate a battle between two characters."""

    def __init__(
            self,
            character1: Character,
            character2: Character,
    ) -> None:
        self.character1 = character1
        self.character2 = character2

    def play_round(self) -> None: 
        """Play one round of combat."""
        if self.character1.is_alive;
            self.character1.attack(self.character2)

        if self.character2.is_alive;
            self.character2.attack(self.character1)

    def simulate_battle(self) -> Character:
        """
        Simulate rounds until one character dies.
        Returns:
            Character: winning character.
        """

        round_number: int = 1

        while self.character1.is_alive and self.character2.is_alive:
            print(f"\n--- Round {round_number} ---")

            self.play_round()

            print(
                
            )